# EA-TTM — Final Results and Discussion

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 12 — Definitive Results and Discussion  
**Scope:** Consolidate the locked final-test evidence and the 28-day horizon analysis into one authoritative Results and Discussion section for the final report.

This phase performs **no training, tuning, model selection, checkpoint switching, threshold modification, or test re-evaluation**.

## Governance constraint

The model-development and final-evaluation locks remain binding:

$$
\text{Post-test tuning}=\text{False}.
$$

Phase 12 may only derive descriptive summaries from already-saved locked results and convert them into report-ready interpretation.

## Reporting logic

The section answers six questions:

1. How much did pre-trained TTM improve over seasonal naive?
2. What additional value did head probing provide?
3. Did the locked primary EA-TTM improve over frozen HP-TTM?
4. Was the improvement concentrated in extreme-demand regimes?
5. What do the gate and ablations reveal about the mechanism?
6. How stable was performance across the 28-day horizon?

The final interpretation must retain all limitations. In particular, the ungated cross-feed ablation must be reported because it achieved lower overall and extreme MSE than the locked gated primary model.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

## Project preflight and locked-state verification

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")
METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_TABLE_DIR = RESULTS_DIR / "tables"
RESULTS_FIGURE_DIR = RESULTS_DIR / "figures"

required_artifacts = {
    "phase10_summary": METADATA_DIR / "10_RUN_SUMMARY.json",
    "phase11_manifest": METADATA_DIR / "11_REPORTING_MANIFEST.json",
    "phase11b_manifest": METADATA_DIR / "11B_HORIZON_VISUALIZATION_MANIFEST.json",
    "development_lock": METADATA_DIR / "09_MODEL_DEVELOPMENT_LOCK.json",
    "final_evaluation_lock": METADATA_DIR / "10_FINAL_EVALUATION_LOCK.json",
    "master_results": RESULTS_TABLE_DIR / "11_MASTER_RESULTS_TABLE.csv",
    "horizon_metrics": RESULTS_TABLE_DIR / "11B_FINAL_TEST_HORIZON_METRICS.csv",
    "primary_vs_hp_horizon": RESULTS_TABLE_DIR / "11B_PRIMARY_VS_HP_HORIZON_COMPARISON.csv",
}

missing = [str(p) for p in required_artifacts.values() if not p.exists()]
assert not missing, "Phase 12 prerequisites are incomplete: " + "; ".join(missing)

def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))

phase10 = load_json(METADATA_DIR / "10_RUN_SUMMARY.json")
phase11_manifest = load_json(METADATA_DIR / "11_REPORTING_MANIFEST.json")
phase11b_manifest = load_json(METADATA_DIR / "11B_HORIZON_VISUALIZATION_MANIFEST.json")
development_lock = load_json(METADATA_DIR / "09_MODEL_DEVELOPMENT_LOCK.json")
final_evaluation_lock = load_json(METADATA_DIR / "10_FINAL_EVALUATION_LOCK.json")

assert development_lock["post_lock_tuning_permitted"] is False
assert final_evaluation_lock["post_test_tuning_permitted"] is False
assert final_evaluation_lock["test_results_may_change_model_selection"] is False
assert phase11_manifest["post_test_tuning_permitted"] is False
assert phase11b_manifest["post_test_tuning_permitted"] is False

PRIMARY_MODEL = development_lock["primary_model"]
PRIMARY_EPOCH = int(development_lock["primary_epoch"])

assert PRIMARY_MODEL == "Phase 08 best-balanced EA-TTM"
assert PRIMARY_EPOCH == 15

DATASET_SHA256 = phase10["dataset_sha256"]
SPLIT_SHA256 = phase10["split_sha256"]
DEVELOPMENT_LOCK_SHA256 = phase10["development_lock_sha256"]
FINAL_EVALUATION_LOCK_SHA256 = phase10["final_evaluation_lock_sha256"]

print("Primary model:", PRIMARY_MODEL)
print("Primary epoch:", PRIMARY_EPOCH)
print("Final-test interval:", phase10["final_test_interval"])
print("Post-test tuning permitted:", False)

Mounted at /content/drive
Primary model: Phase 08 best-balanced EA-TTM
Primary epoch: 15
Final-test interval: ['d_1914', 'd_1941']
Post-test tuning permitted: False


## Derive authoritative model and horizon statistics

In [3]:
master_results = pd.read_csv(
    RESULTS_TABLE_DIR / "11_MASTER_RESULTS_TABLE.csv"
)

horizon_metrics = pd.read_csv(
    RESULTS_TABLE_DIR / "11B_FINAL_TEST_HORIZON_METRICS.csv"
)

h = pd.read_csv(
    RESULTS_TABLE_DIR / "11B_PRIMARY_VS_HP_HORIZON_COMPARISON.csv"
)

results = master_results.set_index("model").to_dict(orient="index")

for required_model in [
    "Weekly Seasonal Naive",
    "Zero-shot TTM",
    "Frozen HP-TTM",
    "Retail-only gated adapter",
    "Ungated cross-feed adapter",
    "Primary EA-TTM",
]:
    assert required_model in results

assert set(h["horizon"].astype(int)) == set(range(1, 29))

seasonal = results["Weekly Seasonal Naive"]
zero_shot = results["Zero-shot TTM"]
hp = results["Frozen HP-TTM"]
retail_only = results["Retail-only gated adapter"]
ungated = results["Ungated cross-feed adapter"]
ea = results["Primary EA-TTM"]

def improvement_percent(baseline: float, candidate: float) -> float:
    return float(100.0 * (baseline - candidate) / baseline)

headline = {
    "zero_shot_vs_seasonal_mse_percent": improvement_percent(seasonal["mse"], zero_shot["mse"]),
    "hp_vs_zero_shot_mse_percent": improvement_percent(zero_shot["mse"], hp["mse"]),
    "ea_vs_hp_mse_percent": improvement_percent(hp["mse"], ea["mse"]),
    "ea_vs_hp_rmse_percent": improvement_percent(hp["rmse"], ea["rmse"]),
    "ea_vs_hp_mae_percent": improvement_percent(hp["mae"], ea["mae"]),
    "ea_vs_hp_wape_percent": improvement_percent(hp["wape"], ea["wape"]),
    "ea_vs_hp_normal_mse_percent": improvement_percent(hp["normal_mse"], ea["normal_mse"]),
    "ea_vs_hp_extreme_mse_percent": improvement_percent(hp["extreme_mse"], ea["extreme_mse"]),
    "ea_vs_hp_positive_extreme_mse_percent": improvement_percent(hp["positive_extreme_mse"], ea["positive_extreme_mse"]),
    "ea_vs_hp_negative_extreme_mse_percent": improvement_percent(hp["negative_extreme_mse"], ea["negative_extreme_mse"]),
    "ungated_vs_ea_mse_percent": improvement_percent(ea["mse"], ungated["mse"]),
    "ungated_vs_ea_extreme_mse_percent": improvement_percent(ea["extreme_mse"], ungated["extreme_mse"]),
}

horizon_summary = {
    "overall_mse_improved_horizons": int((h["mse_improvement_percent"] > 0).sum()),
    "extreme_mse_improved_horizons": int((h["extreme_mse_improvement_percent"] > 0).sum()),
    "positive_extreme_mse_improved_horizons": int((h["positive_extreme_mse_improvement_percent"] > 0).sum()),
    "negative_extreme_mse_improved_horizons": int((h["negative_extreme_mse_improvement_percent"] > 0).sum()),
    "mae_improved_horizons": int((h["mae_improvement_percent"] > 0).sum()),
    "wape_improved_horizons": int((h["wape_improvement_percent"] > 0).sum()),
    "best_overall_horizon": int(h.loc[h["mse_improvement_percent"].idxmax(), "horizon"]),
    "best_overall_improvement_percent": float(h["mse_improvement_percent"].max()),
    "worst_overall_horizon": int(h.loc[h["mse_improvement_percent"].idxmin(), "horizon"]),
    "worst_overall_improvement_percent": float(h["mse_improvement_percent"].min()),
    "best_extreme_horizon": int(h.loc[h["extreme_mse_improvement_percent"].idxmax(), "horizon"]),
    "best_extreme_improvement_percent": float(h["extreme_mse_improvement_percent"].max()),
    "worst_extreme_horizon": int(h.loc[h["extreme_mse_improvement_percent"].idxmin(), "horizon"]),
    "worst_extreme_improvement_percent": float(h["extreme_mse_improvement_percent"].min()),
}

week_records = []
for start, end, label in [
    (1, 7, "Days 1-7"),
    (8, 14, "Days 8-14"),
    (15, 21, "Days 15-21"),
    (22, 28, "Days 22-28"),
]:
    block = h.loc[h["horizon"].between(start, end)]
    week_records.append({
        "period": label,
        "start_horizon": start,
        "end_horizon": end,
        "mean_overall_mse_improvement_percent": float(block["mse_improvement_percent"].mean()),
        "mean_extreme_mse_improvement_percent": float(block["extreme_mse_improvement_percent"].mean()),
        "mean_positive_extreme_mse_improvement_percent": float(block["positive_extreme_mse_improvement_percent"].mean()),
        "mean_negative_extreme_mse_improvement_percent": float(block["negative_extreme_mse_improvement_percent"].mean()),
        "mean_mae_improvement_percent": float(block["mae_improvement_percent"].mean()),
        "mean_wape_improvement_percent": float(block["wape_improvement_percent"].mean()),
    })

weekly_horizon_summary = pd.DataFrame(week_records)

association_summary = {
    "extreme_prevalence_vs_extreme_mse_improvement_correlation": float(
        np.corrcoef(h["extreme_count"] / 30490.0, h["extreme_mse_improvement_percent"])[0, 1]
    ),
}

mechanism = phase10["primary_mechanism"]
bootstrap = phase10["paired_bootstrap"]

print(json.dumps({
    "headline": headline,
    "horizon_summary": horizon_summary,
    "association_summary": association_summary,
}, indent=2))

{
  "headline": {
    "zero_shot_vs_seasonal_mse_percent": 35.27758087728278,
    "hp_vs_zero_shot_mse_percent": 1.6130376202844876,
    "ea_vs_hp_mse_percent": 0.7546945937067596,
    "ea_vs_hp_rmse_percent": 0.37806195104752865,
    "ea_vs_hp_mae_percent": -0.8763542193948493,
    "ea_vs_hp_wape_percent": -0.8763542193948418,
    "ea_vs_hp_normal_mse_percent": -0.7322761714554105,
    "ea_vs_hp_extreme_mse_percent": 1.8591384073103026,
    "ea_vs_hp_positive_extreme_mse_percent": 2.1290169444618865,
    "ea_vs_hp_negative_extreme_mse_percent": -1.9247710198155024,
    "ungated_vs_ea_mse_percent": 0.33932646407725825,
    "ungated_vs_ea_extreme_mse_percent": 2.3805775707039136
  },
  "horizon_summary": {
    "overall_mse_improved_horizons": 27,
    "extreme_mse_improved_horizons": 27,
    "positive_extreme_mse_improved_horizons": 26,
    "negative_extreme_mse_improved_horizons": 8,
    "mae_improved_horizons": 0,
    "wape_improved_horizons": 0,
    "best_overall_horizon": 18,
    "be

## Persist definitive report tables

In [4]:
FINAL_REPORT_TABLE_PATH = RESULTS_TABLE_DIR / "12_FINAL_REPORT_RESULTS_TABLE.csv"
WEEKLY_HORIZON_PATH = RESULTS_TABLE_DIR / "12_HORIZON_WEEKLY_SUMMARY.csv"
FIGURE_PLAN_PATH = RESULTS_TABLE_DIR / "12_FIGURE_PLACEMENT_PLAN.csv"
DEFENSE_RESULTS_PATH = RESULTS_TABLE_DIR / "12_DEFENSE_RESULTS_CLAIMS.csv"

master_results[[
    "model", "mse", "rmse", "mae", "wape",
    "normal_mse", "extreme_mse",
    "positive_extreme_mse", "negative_extreme_mse",
]].to_csv(FINAL_REPORT_TABLE_PATH, index=False)

weekly_horizon_summary.to_csv(WEEKLY_HORIZON_PATH, index=False)

figure_plan = pd.DataFrame([
    {"order": 1, "figure": "11_FIG01_FINAL_TEST_MSE.png", "section": "Benchmark progression", "purpose": "Overall locked-test MSE progression across all models."},
    {"order": 2, "figure": "11_FIG02_NORMAL_EXTREME_MSE.png", "section": "Regime results", "purpose": "Normal versus extreme error trade-off."},
    {"order": 3, "figure": "11B_FIG07_FOCUSED_MSE_HP_EA_UNGATED.png", "section": "Ablation evidence", "purpose": "Focused HP-TTM, primary EA-TTM, and ungated comparison."},
    {"order": 4, "figure": "11B_FIG04_PRIMARY_VS_HP_MSE_IMPROVEMENT.png", "section": "28-day horizon", "purpose": "Overall MSE improvement by horizon."},
    {"order": 5, "figure": "11B_FIG05_PRIMARY_VS_HP_EXTREME_MSE_IMPROVEMENT.png", "section": "28-day horizon", "purpose": "Extreme-MSE improvement by horizon."},
    {"order": 6, "figure": "11B_FIG06_EXTREME_PREVALENCE_BY_HORIZON.png", "section": "28-day horizon", "purpose": "Extreme prevalence context by horizon."},
    {"order": 7, "figure": "11_FIG05_BOOTSTRAP_EA_VS_HP.png", "section": "Robustness", "purpose": "Paired bootstrap interval."},
])
figure_plan.to_csv(FIGURE_PLAN_PATH, index=False)

claims = pd.DataFrame([
    {
        "claim": "EA-TTM improved locked-test overall MSE over HP-TTM.",
        "evidence": f"{hp['mse']:.4f} to {ea['mse']:.4f} ({headline['ea_vs_hp_mse_percent']:.2f}% improvement).",
        "qualification": "MAE and WAPE worsened.",
    },
    {
        "claim": "EA-TTM's strongest regime-level evidence is on positive extremes.",
        "evidence": f"Positive-extreme MSE improved by {headline['ea_vs_hp_positive_extreme_mse_percent']:.2f}%.",
        "qualification": f"Negative-extreme MSE changed by {headline['ea_vs_hp_negative_extreme_mse_percent']:.2f}%.",
    },
    {
        "claim": "The gate learned meaningful extreme-regime information.",
        "evidence": f"AUROC={mechanism['gate_auroc']:.3f}; AUPRC lift={mechanism['gate_auprc_lift_vs_prevalence']:.2f}x; effective correction extreme/normal={mechanism['effective_correction_extreme_to_normal_ratio']:.2f}x.",
        "qualification": "The ungated cross-feed ablation achieved lower MSE.",
    },
    {
        "claim": "The squared-error benefit is distributed across most forecast horizons.",
        "evidence": f"Overall MSE improved on {horizon_summary['overall_mse_improved_horizons']}/28 horizons; extreme MSE on {horizon_summary['extreme_mse_improved_horizons']}/28.",
        "qualification": "Magnitude varies across the horizon.",
    },
])
claims.to_csv(DEFENSE_RESULTS_PATH, index=False)

figure_plan

,order,figure,section,purpose
0,1,11_FIG01_FINAL_TEST_MSE.png,Benchmark progression,Overall locked-test MSE progression across all...
1,2,11_FIG02_NORMAL_EXTREME_MSE.png,Regime results,Normal versus extreme error trade-off.
2,3,11B_FIG07_FOCUSED_MSE_HP_EA_UNGATED.png,Ablation evidence,"Focused HP-TTM, primary EA-TTM, and ungated co..."
3,4,11B_FIG04_PRIMARY_VS_HP_MSE_IMPROVEMENT.png,28-day horizon,Overall MSE improvement by horizon.
4,5,11B_FIG05_PRIMARY_VS_HP_EXTREME_MSE_IMPROVEMEN...,28-day horizon,Extreme-MSE improvement by horizon.
5,6,11B_FIG06_EXTREME_PREVALENCE_BY_HORIZON.png,28-day horizon,Extreme prevalence context by horizon.
6,7,11_FIG05_BOOTSTRAP_EA_VS_HP.png,Robustness,Paired bootstrap interval.


## Generate the authoritative Results and Discussion narrative

In [5]:
def signed_phrase(value: float, noun: str) -> str:
    if value >= 0:
        return f"{noun} improved by {value:.2f}%"
    return f"{noun} worsened by {abs(value):.2f}%"

best_week = weekly_horizon_summary.loc[
    weekly_horizon_summary["mean_overall_mse_improvement_percent"].idxmax()
]
best_extreme_week = weekly_horizon_summary.loc[
    weekly_horizon_summary["mean_extreme_mse_improvement_percent"].idxmax()
]

md_lines = [
    "# Final Results and Discussion",
    "",
    "## 1. Benchmark progression",
    "",
    (
        f"The weekly seasonal-naive benchmark produced a locked final-test MSE of {seasonal['mse']:.4f}, "
        f"RMSE of {seasonal['rmse']:.4f}, MAE of {seasonal['mae']:.4f}, and WAPE of {seasonal['wape']:.4f}. "
        f"Zero-shot TTM reduced MSE to {zero_shot['mse']:.4f}, equivalent to a "
        f"{headline['zero_shot_vs_seasonal_mse_percent']:.2f}% reduction in squared error. "
        "This demonstrates substantial transfer from the pre-trained TTM before any retail-domain parameter update."
    ),
    "",
    (
        f"Head probing further reduced final-test MSE from {zero_shot['mse']:.4f} to {hp['mse']:.4f}, "
        f"an additional {headline['hp_vs_zero_shot_mse_percent']:.2f}% improvement. "
        "The gain was smaller than the initial zero-shot improvement but established a stronger frozen retail-adapted base model."
    ),
    "",
    "## 2. Primary EA-TTM performance",
    "",
    (
        f"The locked primary EA-TTM achieved MSE {ea['mse']:.4f} compared with {hp['mse']:.4f} for frozen HP-TTM, "
        f"a {headline['ea_vs_hp_mse_percent']:.2f}% MSE improvement and a {headline['ea_vs_hp_rmse_percent']:.2f}% RMSE improvement."
    ),
    "",
    (
        f"The improvement was not universal across metrics: {signed_phrase(headline['ea_vs_hp_mae_percent'], 'MAE')} and "
        f"{signed_phrase(headline['ea_vs_hp_wape_percent'], 'WAPE')}. This divergence indicates that EA-TTM primarily reduced "
        "comparatively large errors rather than uniformly reducing absolute error."
    ),
    "",
    "## 3. Normal and extreme regimes",
    "",
    (
        f"EA-TTM reduced all-extreme MSE by {headline['ea_vs_hp_extreme_mse_percent']:.2f}% and positive-extreme MSE by "
        f"{headline['ea_vs_hp_positive_extreme_mse_percent']:.2f}%. By contrast, "
        f"{signed_phrase(headline['ea_vs_hp_normal_mse_percent'], 'normal-regime MSE')} and "
        f"{signed_phrase(headline['ea_vs_hp_negative_extreme_mse_percent'], 'negative-extreme MSE')}."
    ),
    "",
    (
        f"The final-test extreme rate was {100.0 * mechanism['test_extreme_rate']:.2f}%. Positive extremes represented "
        f"{100.0 * mechanism['test_positive_extreme_rate']:.2f}% of all observations, while negative extremes represented only "
        f"{100.0 * mechanism['test_negative_extreme_rate']:.2f}%. The strongest empirical support is therefore for positive demand surges, "
        "not symmetric improvement across upward and downward shocks."
    ),
    "",
    "## 4. Gate mechanism",
    "",
    (
        f"The gate achieved AUROC {mechanism['gate_auroc']:.3f} and AUPRC {mechanism['gate_auprc']:.3f}. Its AUPRC was "
        f"{mechanism['gate_auprc_lift_vs_prevalence']:.2f} times the extreme prevalence. Mean gate probability was "
        f"{mechanism['mean_gate_extreme']:.3f} on extreme observations and {mechanism['mean_gate_normal']:.3f} on normal observations."
    ),
    "",
    (
        f"The mean absolute effective correction |g×Delta| was {mechanism['effective_correction_extreme_to_normal_ratio']:.2f} times larger "
        f"for extreme observations, and correction direction agreed with the required residual direction on "
        f"{100.0 * mechanism['extreme_correction_direction_agreement']:.2f}% of extreme observations. This demonstrates meaningful routing, "
        "but mechanism quality is not equivalent to forecasting optimality."
    ),
    "",
    "## 5. Ablation evidence",
    "",
    (
        f"The retail-only gated adapter achieved MSE {retail_only['mse']:.4f}, while the full primary EA-TTM achieved {ea['mse']:.4f}. "
        "Together with the validation ablation, this supports a small incremental contribution from the decoder cross-feed."
    ),
    "",
    (
        f"The ungated cross-feed adapter achieved MSE {ungated['mse']:.4f} and extreme MSE {ungated['extreme_mse']:.4f}, both lower than "
        f"the primary EA-TTM values of {ea['mse']:.4f} and {ea['extreme_mse']:.4f}. Relative to the primary model, the ungated ablation "
        f"reduced overall MSE by {headline['ungated_vs_ea_mse_percent']:.2f}% and extreme MSE by {headline['ungated_vs_ea_extreme_mse_percent']:.2f}%."
    ),
    "",
    (
        "This limits the strongest architectural claim: decoder cross-feed and residual correction are useful, but the multiplicative gate is not "
        "the forecast-optimal component under these experiments. The primary model was not replaced because it had been locked before final-test access."
    ),
    "",
    "## 6. Bootstrap robustness",
    "",
    (
        f"The paired series-level MSE difference (HP-TTM minus EA-TTM) was {bootstrap['observed_mse_difference_hp_minus_ea']:.5f}. "
        f"Across {int(bootstrap['replicates']):,} bootstrap replicates, the 95% percentile interval was "
        f"[{bootstrap['ci_95_lower']:.5f}, {bootstrap['ci_95_upper']:.5f}], entirely above zero under the chosen store-item resampling scheme."
    ),
    "",
    "## 7. Performance across the 28-day horizon",
    "",
    (
        f"EA-TTM reduced overall MSE relative to HP-TTM on {horizon_summary['overall_mse_improved_horizons']} of 28 forecast steps and "
        f"reduced extreme MSE on {horizon_summary['extreme_mse_improved_horizons']} of 28. The strongest overall horizon improvement occurred at "
        f"horizon {horizon_summary['best_overall_horizon']} ({horizon_summary['best_overall_improvement_percent']:.2f}%), while the weakest occurred "
        f"at horizon {horizon_summary['worst_overall_horizon']} ({horizon_summary['worst_overall_improvement_percent']:.2f}%)."
    ),
    "",
    (
        f"The strongest extreme-MSE improvement occurred at horizon {horizon_summary['best_extreme_horizon']} "
        f"({horizon_summary['best_extreme_improvement_percent']:.2f}%). The largest mean overall MSE improvement occurred during "
        f"{best_week['period']} ({best_week['mean_overall_mse_improvement_percent']:.2f}%), while the largest mean extreme-MSE improvement occurred "
        f"during {best_extreme_week['period']} ({best_extreme_week['mean_extreme_mse_improvement_percent']:.2f}%)."
    ),
    "",
    (
        f"Positive-extreme MSE improved on {horizon_summary['positive_extreme_mse_improved_horizons']} horizons, whereas negative-extreme MSE improved "
        f"on only {horizon_summary['negative_extreme_mse_improved_horizons']}. MAE improved on {horizon_summary['mae_improved_horizons']} horizons and "
        f"WAPE on {horizon_summary['wape_improved_horizons']}. This reinforces the aggregate finding that EA-TTM mainly reduces squared error in the "
        "positive tail rather than uniformly improving absolute error."
    ),
    "",
    (
        f"The descriptive correlation between horizon-specific extreme prevalence and extreme-MSE improvement was "
        f"{association_summary['extreme_prevalence_vs_extreme_mse_improvement_correlation']:.2f}. This is treated as association only, not causation."
    ),
    "",
    "## 8. Overall interpretation",
    "",
    (
        "The evidence supports a qualified conclusion. Pre-trained TTM transferred strongly to retail forecasting, head probing added further retail adaptation, "
        "and the locked EA-TTM produced a modest but broadly distributed reduction in squared error with stronger gains on positive extreme-demand errors. "
        "The gate learned meaningful extreme-regime behavior, but the ungated cross-feed ablation achieved lower overall and extreme MSE. The contribution is "
        "therefore best framed as evidence for lightweight decoder cross-feed and retail-aware residual adaptation under tight compute constraints, rather than "
        "universal metric superiority or proof that multiplicative gating is uniquely optimal."
    ),
]

RESULTS_DISCUSSION_MD_PATH = RESULTS_DIR / "12_RESULTS_AND_DISCUSSION.md"
RESULTS_DISCUSSION_MD_PATH.write_text("\n".join(md_lines), encoding="utf-8")

print("Saved:", RESULTS_DISCUSSION_MD_PATH)

Saved: /content/drive/MyDrive/EA_TTM_RETAIL/results/12_RESULTS_AND_DISCUSSION.md


## Generate LaTeX-ready section

In [6]:
RESULTS_DISCUSSION_TEX_PATH = RESULTS_DIR / "12_RESULTS_AND_DISCUSSION.tex"

tex = rf"""\section{{Results and Discussion}}

\subsection{{Benchmark Progression}}
The weekly seasonal-naive benchmark achieved a locked final-test MSE of {seasonal['mse']:.4f}, RMSE of {seasonal['rmse']:.4f}, MAE of {seasonal['mae']:.4f}, and WAPE of {seasonal['wape']:.4f}. Zero-shot TTM reduced MSE to {zero_shot['mse']:.4f}, corresponding to a {headline['zero_shot_vs_seasonal_mse_percent']:.2f}\% reduction in squared error. Head probing further reduced MSE to {hp['mse']:.4f}, an additional {headline['hp_vs_zero_shot_mse_percent']:.2f}\% improvement over zero-shot TTM.

\subsection{{Primary EA-TTM Performance}}
The locked primary EA-TTM achieved MSE {ea['mse']:.4f} compared with {hp['mse']:.4f} for frozen HP-TTM, corresponding to a {headline['ea_vs_hp_mse_percent']:.2f}\% MSE improvement and a {headline['ea_vs_hp_rmse_percent']:.2f}\% RMSE improvement. MAE changed by {headline['ea_vs_hp_mae_percent']:.2f}\% and WAPE changed by {headline['ea_vs_hp_wape_percent']:.2f}\%, where negative values denote deterioration. The result therefore indicates reduced large-error exposure rather than universal improvement in absolute error.

\subsection{{Regime-Specific Performance}}
EA-TTM improved all-extreme MSE by {headline['ea_vs_hp_extreme_mse_percent']:.2f}\% and positive-extreme MSE by {headline['ea_vs_hp_positive_extreme_mse_percent']:.2f}\%. Normal-regime MSE changed by {headline['ea_vs_hp_normal_mse_percent']:.2f}\%, and negative-extreme MSE changed by {headline['ea_vs_hp_negative_extreme_mse_percent']:.2f}\%. Positive extremes accounted for {100.0 * mechanism['test_positive_extreme_rate']:.2f}\% of all final-test observations, while negative extremes accounted for only {100.0 * mechanism['test_negative_extreme_rate']:.2f}\%.

\subsection{{Gate Mechanism}}
The Extreme Gate achieved AUROC {mechanism['gate_auroc']:.3f} and AUPRC {mechanism['gate_auprc']:.3f}, representing a {mechanism['gate_auprc_lift_vs_prevalence']:.2f}$\times$ lift over the extreme prevalence. The mean absolute effective correction $|g\Delta|$ was {mechanism['effective_correction_extreme_to_normal_ratio']:.2f}$\times$ larger for extreme observations, and correction direction agreed with the required residual direction on {100.0 * mechanism['extreme_correction_direction_agreement']:.2f}\% of extreme observations.

\subsection{{Ablation Evidence}}
The retail-only gated adapter achieved MSE {retail_only['mse']:.4f}, compared with {ea['mse']:.4f} for the full primary EA-TTM. The ungated cross-feed adapter achieved MSE {ungated['mse']:.4f} and extreme MSE {ungated['extreme_mse']:.4f}, both lower than the locked primary EA-TTM. Relative to the primary model, the ungated adapter reduced overall MSE by {headline['ungated_vs_ea_mse_percent']:.2f}\% and extreme MSE by {headline['ungated_vs_ea_extreme_mse_percent']:.2f}\%. This supports the decoder cross-feed and residual pathway while showing that multiplicative gating is not forecast-optimal in this experiment. The primary model was not changed because model selection was locked before final-test access.

\subsection{{Bootstrap Robustness}}
The observed paired series-level MSE difference (HP-TTM minus EA-TTM) was {bootstrap['observed_mse_difference_hp_minus_ea']:.5f}. Across {int(bootstrap['replicates']):,} paired bootstrap replicates, the 95\% percentile interval was [{bootstrap['ci_95_lower']:.5f}, {bootstrap['ci_95_upper']:.5f}], remaining above zero under the chosen store-item resampling scheme.

\subsection{{Performance Across the 28-Day Horizon}}
EA-TTM reduced overall MSE on {horizon_summary['overall_mse_improved_horizons']} of 28 horizons and extreme MSE on {horizon_summary['extreme_mse_improved_horizons']} of 28. The strongest overall improvement occurred at horizon {horizon_summary['best_overall_horizon']} ({horizon_summary['best_overall_improvement_percent']:.2f}\%), and the strongest extreme-MSE improvement occurred at horizon {horizon_summary['best_extreme_horizon']} ({horizon_summary['best_extreme_improvement_percent']:.2f}\%). The largest mean overall MSE improvement occurred during {best_week['period']} ({best_week['mean_overall_mse_improvement_percent']:.2f}\%), while the largest mean extreme-MSE improvement occurred during {best_extreme_week['period']} ({best_extreme_week['mean_extreme_mse_improvement_percent']:.2f}\%). Positive-extreme MSE improved on {horizon_summary['positive_extreme_mse_improved_horizons']} horizons, whereas negative-extreme MSE improved on only {horizon_summary['negative_extreme_mse_improved_horizons']} horizons.

\subsection{{Overall Interpretation}}
The evidence supports a qualified conclusion: lightweight decoder cross-feed and retail-aware residual adaptation can improve the squared-error and positive-tail behavior of a frozen retail-adapted TTM under tight parameter and compute constraints. The results do not support universal superiority across every metric, nor do they establish multiplicative gating as the uniquely optimal correction mechanism.
"""

RESULTS_DISCUSSION_TEX_PATH.write_text(tex.strip() + "\n", encoding="utf-8")
print("Saved:", RESULTS_DISCUSSION_TEX_PATH)

Saved: /content/drive/MyDrive/EA_TTM_RETAIL/results/12_RESULTS_AND_DISCUSSION.tex


## Phase 12 manifest and project status

In [7]:
PHASE12_SUMMARY_PATH = METADATA_DIR / "12_RUN_SUMMARY.json"

phase12_summary = {
    "status": "PHASE_12_RESULTS_DISCUSSION_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "development_lock_sha256": DEVELOPMENT_LOCK_SHA256,
    "final_evaluation_lock_sha256": FINAL_EVALUATION_LOCK_SHA256,
    "primary_model": PRIMARY_MODEL,
    "primary_epoch": PRIMARY_EPOCH,
    "post_test_tuning_permitted": False,
    "headline": headline,
    "horizon_summary": horizon_summary,
    "weekly_horizon_summary": week_records,
    "association_summary": association_summary,
    "artifacts": {
        "results_discussion_markdown": str(RESULTS_DISCUSSION_MD_PATH),
        "results_discussion_latex": str(RESULTS_DISCUSSION_TEX_PATH),
        "final_report_results_table": str(FINAL_REPORT_TABLE_PATH),
        "weekly_horizon_summary": str(WEEKLY_HORIZON_PATH),
        "figure_placement_plan": str(FIGURE_PLAN_PATH),
        "defense_results_claims": str(DEFENSE_RESULTS_PATH),
    },
}

PHASE12_SUMMARY_PATH.write_text(
    json.dumps(phase12_summary, indent=2),
    encoding="utf-8",
)

status_path = PROJECT_ROOT / "PROJECT_STATUS.md"
status_path.write_text(
    "\n".join([
        "# EA-TTM Project Status",
        "",
        "## Current stage",
        "Phase 12 — Definitive Results and Discussion completed.",
        "",
        "## Experimental governance",
        "- Model development: LOCKED",
        "- Final test: COMPLETE",
        "- Post-test tuning: PROHIBITED",
        "",
        "## Primary model",
        f"- {PRIMARY_MODEL}",
        f"- Epoch: {PRIMARY_EPOCH}",
        "",
        "## Next phase",
        "Phase 13 — Assemble and compile the complete final LaTeX report.",
    ]),
    encoding="utf-8",
)


349

## Return block

In [8]:
return_block = {
    "status": "PHASE_12_RESULTS_DISCUSSION_COMPLETE",
    "primary_model": PRIMARY_MODEL,
    "primary_epoch": PRIMARY_EPOCH,
    "post_test_tuning_permitted": False,
    "headline": headline,
    "horizon_summary": horizon_summary,
    "weekly_horizon_summary": week_records,
    "association_summary": association_summary,
    "artifacts": phase12_summary["artifacts"],
}

print("PHASE 12 RETURN BLOCK")
print(json.dumps(return_block, indent=2))

PHASE 12 RETURN BLOCK
{
  "status": "PHASE_12_RESULTS_DISCUSSION_COMPLETE",
  "primary_model": "Phase 08 best-balanced EA-TTM",
  "primary_epoch": 15,
  "post_test_tuning_permitted": false,
  "headline": {
    "zero_shot_vs_seasonal_mse_percent": 35.27758087728278,
    "hp_vs_zero_shot_mse_percent": 1.6130376202844876,
    "ea_vs_hp_mse_percent": 0.7546945937067596,
    "ea_vs_hp_rmse_percent": 0.37806195104752865,
    "ea_vs_hp_mae_percent": -0.8763542193948493,
    "ea_vs_hp_wape_percent": -0.8763542193948418,
    "ea_vs_hp_normal_mse_percent": -0.7322761714554105,
    "ea_vs_hp_extreme_mse_percent": 1.8591384073103026,
    "ea_vs_hp_positive_extreme_mse_percent": 2.1290169444618865,
    "ea_vs_hp_negative_extreme_mse_percent": -1.9247710198155024,
    "ungated_vs_ea_mse_percent": 0.33932646407725825,
    "ungated_vs_ea_extreme_mse_percent": 2.3805775707039136
  },
  "horizon_summary": {
    "overall_mse_improved_horizons": 27,
    "extreme_mse_improved_horizons": 27,
    "positive_e